<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/james-le-twelve-labs/pegasus-qc-playground/blob/main/notebooks/Pegasus_AI_Video_QC.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Colab</a>
  </td>
</table>

# Catch AI video slop with Pegasus — the 5-check QC suite

AI video generators produce clips that are *unreliably good*: warped hands, missing brief elements, dialogue attributed to the wrong character, TTS laid over a mouth that never moves. This notebook builds the five production QC checks that catch all of it, returning structured, machine-readable — and where it matters, **timestamped** — verdicts.

You'll run every check against real, provenance-logged AI-generated clips (nothing staged — model, prompt and seed for each clip are in the [provenance log](https://pegasus-qc-playground.vercel.app/provenance.md)):

| # | Check | Mode | The sample clip it catches |
|---|---|---|---|
| 1 | **Brief-match** | Verdict | Product shot missing three ordered elements |
| 2 | **Content verification** | Verdict | The wrong character says the scripted line |
| 3 | **AV-defect audit** | Timeline | A magician's fingers warp and melt mid-shuffle |
| 4 | **Lip-sync verification** | Timeline | TTS voiceover over a still mouth |
| 5 | **Content log** | Timeline | A clean clip, cataloged scene by scene |

Prefer to see it before building it? The [live QC Playground](https://pegasus-qc-playground.vercel.app?utm_source=colab&utm_medium=notebook&utm_campaign=ai-video-qc) runs all of this in your browser and shows the API call behind every verdict.

## Prerequisites

You need a TwelveLabs API key: [sign up free](https://playground.twelvelabs.io?utm_source=colab&utm_medium=notebook&utm_campaign=ai-video-qc), then copy the key from your dashboard. In Colab, store it as a **Secret** named `TL_API_KEY` (key icon in the left sidebar); locally, export it as the `TL_API_KEY` environment variable.

## 1. Setup

In [ ]:
%pip install -q twelvelabs

In [2]:
import json
import os
import time

from twelvelabs import TwelveLabs
from twelvelabs.types import AsyncResponseFormat, VideoContext_AssetId

In [3]:
TL_API_KEY = None
try:  # Google Colab: read the Secret named TL_API_KEY
    from google.colab import userdata
    TL_API_KEY = userdata.get("TL_API_KEY")
except ImportError:  # anywhere else: env var, then prompt
    TL_API_KEY = os.environ.get("TL_API_KEY")
if not TL_API_KEY:
    from getpass import getpass
    TL_API_KEY = getpass("Paste your TwelveLabs API key: ")

client = TwelveLabs(api_key=TL_API_KEY)

## 2. Upload the sample clips as assets

Pegasus 1.5 works directly on **assets** — no index required. Upload once, reuse the `asset_id` for every check. (Swap in your own clip URLs anytime; each upload takes a few seconds.)

In [4]:
DEMO = "https://pegasus-qc-playground.vercel.app"

CLIPS = {
    "talking-head":  f"{DEMO}/clips/talking-head.mp4",   # TTS laid over a still mouth
    "product-broll": f"{DEMO}/clips/product-broll.mp4",  # misses three brief elements
    "card-shuffle":  f"{DEMO}/clips/card-shuffle.mp4",   # warping / malformed hands
    "dialogue":      f"{DEMO}/clips/dialogue.mp4",       # wrong character says the line
    "beach-dog":     f"{DEMO}/clips/beach-dog.mp4",      # the clean one
}

def upload(url):
    asset = client.assets.create(method="url", url=url)
    while client.assets.retrieve(asset.id).status != "ready":
        time.sleep(3)
    return asset.id

ASSETS = {name: upload(url) for name, url in CLIPS.items()}
for name, asset_id in ASSETS.items():
    print(f"ready  {name}  →  {asset_id}")

ready  talking-head  →  6a4c6b6b0d0dea76ce39b7cb
ready  product-broll  →  6a4c6b6e9dc43a0535722d43
ready  card-shuffle  →  6a4c6b70c7babe8a7abbb14f
ready  dialogue  →  6a4c6b7397b9fceedb29bb41
ready  beach-dog  →  6a4c6b7697b9fceedb29bb45


## 3. Two modes, one decision

Pegasus 1.5 gives you two ways to run QC — picking the right one is the only architectural decision here:

| | **Verdict** — prompt-based `analyze` | **Timeline** — Segment / `time_based_metadata` |
|---|---|---|
| You send | An engineered prompt | A segment *schema* (no prompt) |
| You get | One JSON answer for the whole clip | Timestamped segments with typed fields |
| Best for | "Did I get what I ordered?" | "Where exactly is it broken?" |

Rule of thumb: QC **over time** almost always wants Timeline — a timestamped FAIL tells you what to trim, seek, or regenerate instead of re-rolling the whole clip.

## 4. Verdict check — Brief-match ("did I get what I ordered?")

The highest-volume QC question in production. Send the original brief with the clip and demand strict JSON back. The prompt ends with a hard "Respond with ONLY valid JSON" — keep that; it's what makes the output parseable.

In [5]:
BRIEF_MATCH_PROMPT = """You are a broadcast generation-QA engineer inspecting a SINGLE freshly generated
video clip (one beat) BEFORE it is composited into a final cut. Your only job is
to answer: did the generator produce what was requested?

The clip was ordered with this brief:
"<<BRIEF>>"

Judge ONLY whether the brief was fulfilled. Do NOT score creative quality or taste.

Respond with ONLY valid JSON:
{
  "brief_fulfilled": true/false,
  "subject_present": true/false,
  "action_present": true/false,
  "setting_correct": true/false,
  "missing_or_wrong": ["<each element that is missing or wrong>"],
  "confidence": 0.0-1.0,
  "notes": "<one sentence>"
}"""

def brief_match(asset_id, brief):
    res = client.analyze(
        video=VideoContext_AssetId(asset_id=asset_id),
        model_name="pegasus1.5",
        prompt=BRIEF_MATCH_PROMPT.replace("<<BRIEF>>", brief),
        temperature=0.2,  # low = deterministic, better for QC
    )
    return json.loads(res.data)

In [6]:
# This brief is the EXACT prompt the clip was generated from — so any miss is real generator drift.
brief = ("A red ceramic coffee mug with the word \"SUMMIT\" printed on it in white capital letters, steam rising from the coffee, on a rustic wooden desk next to an open silver laptop and a small white plate holding two croissants, camera slowly pans left to right, warm morning light through a window")

verdict = brief_match(ASSETS["product-broll"], brief)
print("PASS" if verdict["brief_fulfilled"] else "FAIL", f'(confidence {verdict["confidence"]})')
print(json.dumps(verdict, indent=2))

FAIL (confidence 0.95)
{
  "brief_fulfilled": false,
  "subject_present": true,
  "action_present": false,
  "setting_correct": true,
  "missing_or_wrong": [
    "The laptop is closed, not open as requested.",
    "The plate holds only one croissant, not two.",
    "The camera is static; the requested slow pan left to right is missing."
  ],
  "confidence": 0.95,
  "notes": "The generator failed to render the laptop as open, included only one croissant instead of two, and did not execute the requested camera pan."
}


When we ran this, Pegasus flagged three real misses: the laptop rendered **closed** (ordered open), **one** croissant (ordered two), and a static camera (ordered a slow pan). That `missing_or_wrong` array is your automated re-prompt — feed it back to the generator and order exactly what was missed.

**That was your first structured QC verdict.** Everything else is variations.

## 5. Verdict check — Content verification (right character, right line)

Same pattern, different question: in a multi-character clip, did the *right* character deliver the scripted line? Our dialogue clip was generated with the brief "the **woman** says the line" — the generator gave it to the bearded man instead. Let's catch that.

In [7]:
CONTENT_VERIFICATION_PROMPT = """You are verifying dialogue attribution in a multi-character clip.

Expected line: "<<LINE>>"
Expected speaker: "<<SPEAKER>>"

Watch and listen. Confirm the correct character speaks the correct line, and that
the audio matches the script.

Respond with ONLY valid JSON:
{
  "line_spoken_matches_script": true/false,
  "correct_character_speaking": true/false,
  "actual_spoken_words": "<what you HEAR>",
  "actual_speaker": "<who appears to say it>",
  "notes": "<one sentence>"
}"""

def content_verification(asset_id, line, speaker):
    prompt = (CONTENT_VERIFICATION_PROMPT
              .replace("<<LINE>>", line)
              .replace("<<SPEAKER>>", speaker))
    res = client.analyze(
        video=VideoContext_AssetId(asset_id=asset_id),
        model_name="pegasus1.5",
        prompt=prompt,
        temperature=0.2,
    )
    return json.loads(res.data)

verdict = content_verification(
    ASSETS["dialogue"],
    line="The shipment arrives tomorrow at nine.",
    speaker="woman in a denim jacket",
)
ok = verdict["line_spoken_matches_script"] and verdict["correct_character_speaking"]
print("PASS" if ok else "FAIL")
print(json.dumps(verdict, indent=2))

FAIL
{
  "line_spoken_matches_script": true,
  "correct_character_speaking": false,
  "actual_spoken_words": "The shipment arrives tomorrow at nine.",
  "actual_speaker": "man in a maroon vest",
  "notes": "The audio matches the script, but the line is spoken by the man, not the woman in the denim jacket as expected."
}


## 6. Timeline — the AV-defect audit (the centerpiece)

Now the mode shift. Instead of a prompt, Segment mode takes **segment definitions** — a schema describing what to find and what typed fields to extract per hit. Three gotchas before you start:

- **No `prompt` in this mode** — passing one is a 400. Your instructions live in the segment/field `description`s.
- **Every field requires a `description`** — and they are load-bearing (see the lip-sync section below).
- **`finish_reason == "length"`** means truncated JSON — raise `max_tokens` (up to 98,304) or trim fields.

In [8]:
def run_segment_task(asset_id, definition, min_segment_duration=2.0):
    """Create a time_based_metadata task, poll to completion, return its segments."""
    task = client.analyze_async.tasks.create(
        video=VideoContext_AssetId(asset_id=asset_id),
        model_name="pegasus1.5",
        analysis_mode="time_based_metadata",
        temperature=0.2,
        min_segment_duration=min_segment_duration,
        response_format=AsyncResponseFormat(
            type="segment_definitions",
            segment_definitions=[definition],
        ),
    )
    while True:
        task = client.analyze_async.tasks.retrieve(task.task_id)
        if task.status in ("ready", "failed"):
            break
        time.sleep(5)
    if task.status == "failed":
        raise RuntimeError(f"analyze task failed: {task.error}")
    if getattr(task.result, "finish_reason", None) == "length":
        print("⚠ output truncated — raise max_tokens or trim fields")
    return json.loads(task.result.data)[definition["id"]]

In [9]:
AV_DEFECTS = {
    "id": "av_defects",
    "description": "Segment the clip wherever a technical audio-visual defect is visible or audible. Include flickering, visual artifacts, warping or morphing objects, malformed hands or faces, duplicated limbs, sudden resolution or exposure jumps, audio dropout, distortion, or lip/audio sync drift. Do NOT judge creative quality (hook, brand fit, aesthetics).",
    "fields": [
        {
            "name": "defect_type",
            "type": "string",
            "description": "The primary defect in this segment",
            "enum": [
                "flicker",
                "artifact",
                "warping",
                "malformed_anatomy",
                "duplicated_limb",
                "exposure_jump",
                "audio_dropout",
                "distortion",
                "sync_drift",
                "other"
            ]
        },
        {
            "name": "severity",
            "type": "string",
            "description": "How disqualifying the defect is",
            "enum": [
                "minor",
                "major"
            ]
        },
        {
            "name": "description",
            "type": "string",
            "description": "One-sentence description of what is wrong in this segment"
        }
    ]
}

defects = run_segment_task(ASSETS["card-shuffle"], AV_DEFECTS)
print("PASS — no defects" if not defects else f"FAIL — {len(defects)} defect segment(s)")
for seg in defects:
    m = seg["metadata"]
    print(f'{seg["start_time"]:.1f}s–{seg["end_time"]:.1f}s · {m["defect_type"]} · {m["severity"]} — {m["description"]}')

FAIL — 1 defect segment(s)
0.0s–5.0s · malformed_anatomy · major — The hands shuffling the cards exhibit unnatural finger movements and anatomical inconsistencies throughout the clip.


Every entry carries a `start_time` and `end_time` — that's what turns "the clip is bad" into "the clip is bad *from here to here*", which is what you need to trim, seek a reviewer, or regenerate just the broken beat. An empty list means PASS, so a pass is a falsifiable claim.

In the [live demo](https://pegasus-qc-playground.vercel.app?utm_source=colab&utm_medium=notebook&utm_campaign=ai-video-qc) these same segments render as a clickable defect timeline under the player.

## 7. Timeline — Lip-sync verification (and three hard-won schema lessons)

This check catches the most common real-world lip-sync failure: **TTS narration laid over footage of someone who is not speaking.** The schema below took three iterations to get right, and each fix is a general lesson for segment-mode QC:

1. **Field descriptions ARE the prompt.** Our first version ("True if lips articulate the words that are heard") *passed* a clip with a completely closed mouth under TTS audio. Booleans must force visual attention: *"true ONLY if you can SEE the mouth repeatedly open and close…"*
2. **Conditional checks need an applicability gate.** On speechless clips the model returned all-false segments — which naive pass logic reads as a lip-sync FAIL on a clip with no lips. The `speech_audible` field is the gate: only judge `lips_match_words` where it's true.
3. **Field order matters.** With the audio gate placed *first*, answering "speech is audible" anchored a speaking-person prior and the mouth booleans followed it — the broken clip passed again, consistently. Visual judgments go before audio anchors, and every field states which modality to judge.

The full evolution is logged in the [provenance file](https://pegasus-qc-playground.vercel.app/provenance.md).

In [10]:
LIPSYNC = {
    "id": "lipsync",
    "description": "Segment every window where speech is audible. If no human speech is audible anywhere in the clip, return no segments. In each window, watch the on-camera person's MOUTH closely before answering the fields: does the mouth visibly open, close, and articulate in time with each word heard? Common failure to catch: voiceover or TTS audio laid over footage of a person whose mouth is closed, still, or smiling without speaking.",
    "fields": [
        {
            "name": "mouth_visibly_articulating",
            "type": "boolean",
            "description": "Judge the VIDEO only, ignoring the audio: true ONLY if you can SEE the mouth repeatedly open and close in speech-like articulation. False if the lips stay closed, hold a smile, or barely move \u2014 even if speech is heard."
        },
        {
            "name": "lips_match_words",
            "type": "boolean",
            "description": "True ONLY if the visible mouth movements track the specific words heard. False if the mouth is closed or its motion does not correspond to the phonemes."
        },
        {
            "name": "native_lipsync_not_tts_over_silence",
            "type": "boolean",
            "description": "False if the audio sounds like narration/TTS laid over footage of someone who is not speaking on camera. True only if the on-camera person is genuinely producing the words."
        },
        {
            "name": "speech_audible",
            "type": "boolean",
            "description": "Judge the AUDIO TRACK only: true if spoken words are audible in this window, regardless of whether anyone on camera appears to be speaking. False for silent footage, music, or ambient sound without spoken words."
        },
        {
            "name": "spoken_words",
            "type": "string",
            "description": "What you HEAR the speaker say in this window"
        },
        {
            "name": "notes",
            "type": "string",
            "description": "Describe what the mouth is doing during this window and any mismatch or timing issue"
        }
    ]
}

segments = run_segment_task(ASSETS["talking-head"], LIPSYNC)
speaking = [s for s in segments if s["metadata"].get("speech_audible")]
sync_ok = all(s["metadata"].get("lips_match_words") for s in speaking)
print("PASS" if sync_ok else "FAIL")
for seg in segments:
    print(f'{seg["start_time"]:.1f}s–{seg["end_time"]:.1f}s', json.dumps(seg["metadata"], indent=2))

FAIL
0.0s–5.0s {
  "lips_match_words": false,
  "mouth_visibly_articulating": false,
  "native_lipsync_not_tts_over_silence": false,
  "notes": "The woman's mouth remains mostly closed or in a static smile throughout the clip. She does not articulate the words heard in the audio track, indicating a voiceover or TTS narration over footage of a person who is not speaking.",
  "speech_audible": true,
  "spoken_words": "Our new product launches this Friday, and we could not be more excited about it."
}


In [11]:
# The applicability gate in action: a clip with no speech at all is N/A, not a FAIL.
na_segments = run_segment_task(ASSETS["beach-dog"], LIPSYNC)
speaking = [s for s in na_segments if s["metadata"].get("speech_audible")]
print(f"segments returned: {len(na_segments)}, with audible speech: {len(speaking)} → nothing to judge → PASS")

segments returned: 0, with audible speech: 0 → nothing to judge → PASS


## 8. Timeline — Content log (cataloging what you keep)

Segment mode isn't only for defect-hunting. The same call structure populates a content catalog with timestamped, searchable metadata — run it on everything that passes QC.

In [12]:
SCENES = {
    "id": "scenes",
    "description": "Segment the clip into distinct scenes based on changes in setting, subject, or visual composition. For each scene, extract catalog metadata.",
    "fields": [
        {
            "name": "summary",
            "type": "string",
            "description": "One-to-two sentence description of what happens in this scene"
        },
        {
            "name": "setting",
            "type": "string",
            "description": "The location or environment of the scene"
        },
        {
            "name": "characters",
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Characters or people visible in the scene"
        },
        {
            "name": "on_screen_text",
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Any visible on-screen text or captions"
        },
        {
            "name": "tags",
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Short catalog tags for search and retrieval"
        }
    ]
}

scenes = run_segment_task(ASSETS["beach-dog"], SCENES)
for seg in scenes:
    m = seg["metadata"]
    print(f'[{seg["start_time"]:.1f}s–{seg["end_time"]:.1f}s] {m["summary"]}')
    print(f'   setting: {m["setting"]} · tags: {", ".join(m["tags"])}')

[0.0s–8.0s] A golden retriever runs joyfully along a sandy beach at sunset, with waves gently washing ashore and the sun casting a warm glow over the scene.
   setting: A sandy beach at sunset with ocean waves and a clear sky. · tags: dog, golden retriever, beach, sunset, running, ocean, nature, pet


## 9. Turn it into a pipeline

Here's the beat that changes this from a demo into infrastructure. Because Timeline verdicts carry timestamps, you don't re-roll failed clips — you re-roll failed *windows*. This is the exact pattern running in production at generative-video studios doing tens of thousands of QC calls: generate → QC → auto-route failures → composite only green clips, with zero human review in between.

In [13]:
def regenerate(clip_name, retry_window):
    # In your pipeline this calls back into your generator (Fal, Kling, Runway, Veo…)
    print(f"→ would re-roll {clip_name} for window {retry_window[0]:.1f}s–{retry_window[1]:.1f}s")

for seg in defects:  # from the card-shuffle audit above
    if seg["metadata"]["severity"] == "major":
        regenerate("card-shuffle", (seg["start_time"], seg["end_time"]))

→ would re-roll card-shuffle for window 0.0s–5.0s


## Next steps

- **Scale it** — batch your checks; scope long inputs with `start_time`/`end_time` or per-definition `time_ranges`; pack up to 10 definitions × 20 fields into one request.
- **Brand/character consistency** — segment definitions accept up to 4 reference images via `media_sources`, referenced as `<@name>` in descriptions: *"Segment where the on-screen product matches `<@intended_product>`."*
- **Wire it to your generator** — the QC call doesn't care where the clip came from; if it has a URL, it can be checked.
- **Docs** — [Segment videos guide](https://docs.twelvelabs.io/docs/guides/segment-videos) · [Analyze videos guide](https://docs.twelvelabs.io/docs/guides/analyze-videos) · [official Segment quickstart](https://github.com/twelvelabs-io/twelvelabs-developer-experience/blob/main/quickstarts/TwelveLabs_Quickstart_Segment.ipynb)
- **Play** — the [live QC Playground](https://pegasus-qc-playground.vercel.app?utm_source=colab&utm_medium=notebook&utm_campaign=ai-video-qc) runs every check in this notebook against every sample clip, no code required.

*Every clip used here is a real AI generation with model, prompt, and seed logged in the [provenance file](https://pegasus-qc-playground.vercel.app/provenance.md). Optional cleanup: uncomment below to delete the uploaded assets.*

In [14]:
# for asset_id in ASSETS.values():
#     client.assets.delete(asset_id)
# print("assets deleted")